TASK 2 — Building a CNN from Scratch

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

Problem 1: Manual Convolution


In [6]:
def conv2d(image, kernel, stride=1, padding=0):
    H_in, W_in = image.shape
    Kh, Kw     = kernel.shape

    if padding > 0:
        image = np.pad(image,
                       pad_width=((padding, padding), (padding, padding)),
                       mode='constant', constant_values=0)

    H_padded, W_padded = image.shape
    H_out = (H_padded - Kh) // stride + 1
    W_out = (W_padded - Kw) // stride + 1

    feature_map = np.zeros((H_out, W_out), dtype=np.float64)

    for i in range(H_out):
        for j in range(W_out):
            row_start = i * stride
            col_start = j * stride
            patch = image[row_start : row_start + Kh,
                          col_start : col_start + Kw]
            feature_map[i, j] = np.sum(patch * kernel)

    return feature_map

test_image = np.array([
    [3, 1, 0, 2, 4],
    [1, 5, 3, 2, 1],
    [0, 2, 6, 4, 3],
    [2, 3, 1, 5, 2],
    [1, 0, 2, 3, 4]
], dtype=np.float64)

sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float64)

result = conv2d(test_image, sobel_x, stride=1, padding=0)
print("Manual conv2d output:")
print(result)
print("Output shape:", result.shape)

Manual conv2d output:
[[ 7. -3. -3.]
 [13.  3. -7.]
 [ 5.  9.  1.]]
Output shape: (3, 3)


Problem 2 - Output Size Derivation

Formula: Output = floor((Input - Kernel + 2×Padding) / Stride) + 1

(a) Input=28, Kernel=5, Padding=0, Stride=1
    = floor((28 - 5 + 0) / 1) + 1
    = floor(23) + 1
    = 24
    Output size: 24 × 24

(b) Input=28, Kernel=3, Padding=1, Stride=1
    = floor((28 - 3 + 2) / 1) + 1
    = floor(27) + 1
    = 28
    Output size: 28 × 28  (same padding preserves dimensions)

(c) Input=32, Kernel=3, Padding=0, Stride=2
    = floor((32 - 3 + 0) / 2) + 1
    = floor(14.5) + 1
    = 15
    Output size: 15 × 15

(d) Two consecutive layers on 32×32:
    Layer 1: K=3, P=1, S=1
    = floor((32 - 3 + 2) / 1) + 1 = 32

    Layer 2: K=3, P=0, S=1
    = floor((32 - 3 + 0) / 1) + 1 = 30

    Final Output size: 30 × 30

Problem 3: LeNet-5

In [7]:
def build_lenet5():
    model = keras.Sequential([
        layers.Conv2D(6, kernel_size=5, activation='tanh',
                      padding='valid', input_shape=(28, 28, 1)),
        layers.AveragePooling2D(pool_size=2, strides=2),
        layers.Conv2D(16, kernel_size=5, activation='tanh', padding='valid'),
        layers.AveragePooling2D(pool_size=2, strides=2),
        layers.Flatten(),
        layers.Dense(120, activation='tanh'),
        layers.Dense(84, activation='tanh'),
        layers.Dense(10, activation='softmax')
    ], name='LeNet5')
    return model

lenet = build_lenet5()
lenet.summary()

k, c_in, c_out = 5, 1, 6
first_conv_params = (k * k * c_in + 1) * c_out
print(f"\nFirst Conv2D params: {first_conv_params}")
print("Total params:", lenet.count_params())

Model: "LeNet5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 24, 24, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_2             │ (None, 12, 12, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 8, 8, 16)       │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_3             │ (None, 4, 4, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 120)            │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,426 (173.54 KB)

 Trainable params: 44,426 (173.54 KB)

 Non-trainable params: 0 (0.00 B)


First Conv2D params: 156
Total params: 44426


**(c) AvgPooling vs MaxPooling:**

LeNet-5 (1998) used AveragePooling because the belief at the time
was that spatial information should be preserved smoothly. Taking
the average was considered more stable and provided better gradient
flow on the limited hardware of that era.

Today, MaxPooling is far more common because it selects the strongest
activation in each region, discarding weaker responses. This helps
the network focus on the most dominant features like sharp edges and
textures, and experimentally produces better accuracy in modern
deep networks.

**Problem 4: Custom CNN**

**Architecture Sketch:**

Input (32×32×3)
      ↓
[Conv2D 32 filters, 3×3] → BatchNorm → ReLU → MaxPool(2×2) → (16×16×32)
      ↓
[Conv2D 64 filters, 3×3] → BatchNorm → ReLU → MaxPool(2×2) → (8×8×64)
      ↓
[Conv2D 128 filters, 3×3] → BatchNorm → ReLU → MaxPool(2×2) → (4×4×128)
      ↓
GlobalAveragePooling → (128,)
      ↓
Dense(1024, ReLU) → Dropout(0.4) → Dense(10, Softmax)

**Design Rationale:**
Three convolutional blocks were chosen because CIFAR-10 images require
sufficient depth to learn features progressively from edges to shapes
to objects. Filter counts double across blocks (32→64→128) so that
deeper layers can represent more abstract features. GlobalAveragePooling
replaces Flatten to reduce parameters and limit overfitting. A Dropout
of 0.4 is applied in the classification head as an additional
regularisation measure.

In [10]:
def build_custom_cnn():
    model = keras.Sequential([
        layers.Conv2D(32, 3, padding='same', input_shape=(32, 32, 3)),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, padding='same',),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2),

        layers.Conv2D(128, 3, padding='same',),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2),

        layers.GlobalAveragePooling2D(),
        layers.Dense(768, activation='relu'),
        layers.Dropout(0.35),
        layers.Dense(10, activation='softmax')
    ], name='CustomCNN_CIFAR10')
    return model

custom_cnn = build_custom_cnn()
custom_cnn.summary()
total_params = custom_cnn.count_params()
print(f"\nTotal parameters: {total_params:,}")
print("Valid range check:", 200_000 <= total_params <= 2_000_000)

Model: "CustomCNN_CIFAR10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_13 (Conv2D)              │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_9 (ReLU)                  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_10 (ReLU)                 │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_11 (ReLU)                 │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 768)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 10)             │         7,690 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 200,906 (784.79 KB)

 Trainable params: 200,458 (783.04 KB)

 Non-trainable params: 448 (1.75 KB)


Total parameters: 200,906
Valid range check: True
